In [1]:
import json
import argparse
from pathlib import Path


def extract_notebook(ipynb_path):
    """Parse a single notebook and return its cell content as a list of lines."""
    ipynb = Path(ipynb_path)
    if not ipynb.exists():
        raise FileNotFoundError(f"Notebook not found: {ipynb}")

    with open(ipynb, "r", encoding="utf-8") as f:
        nb = json.load(f)

    lines = []

    # Notebook header
    lines.append(f"\n{'#'*78}")
    lines.append(f"## NOTEBOOK: {ipynb.name}")
    lines.append(f"## PATH: {ipynb.resolve()}")
    lines.append(f"{'#'*78}\n")

    for i, cell in enumerate(nb.get("cells", []), start=1):
        cell_type = cell.get("cell_type", "unknown")
        source = "".join(cell.get("source", []))

        lines.append(f"\n{'─'*78}")
        lines.append(f"### CELL {i}  |  {cell_type.upper()}")
        lines.append(f"{'─'*78}\n")

        if cell_type == "markdown":
            lines.append(source.strip())

        elif cell_type == "code":
            lines.append("```python")
            lines.append(source.rstrip())
            lines.append("```")

            # Extract outputs
            outputs = cell.get("outputs", [])
            if outputs:
                lines.append("\n# --- Output ---")
                for out in outputs:
                    out_type = out.get("output_type", "")

                    if out_type == "stream":
                        stream_name = out.get("name", "stdout")
                        text = "".join(out.get("text", []))
                        lines.append(f"# [{stream_name}]")
                        lines.append(text.rstrip())

                    elif out_type in ("display_data", "execute_result"):
                        data = out.get("data", {})
                        if "text/plain" in data:
                            txt = "".join(data["text/plain"])
                            lines.append(f"# [result]")
                            lines.append(txt)
                        elif "text/html" in data:
                            lines.append(f"# [HTML output — truncated]")
                        elif "image/png" in data:
                            lines.append(f"# [PNG image output]")
                        else:
                            lines.append(f"# [display: {list(data.keys())}]")

                    elif out_type == "error":
                        ename = out.get("ename", "Error")
                        evalue = out.get("evalue", "")
                        lines.append(f"# [ERROR] {ename}: {evalue}")
                        tb = "\n".join(out.get("traceback", []))
                        if tb:
                            lines.append(f"# Traceback:\n{tb}")

        else:
            lines.append(f"# (Unsupported cell type: {cell_type})")
            lines.append(source.strip())

    # Notebook footer
    lines.append(f"\n{'#'*78}")
    lines.append(f"## END OF: {ipynb.name}")
    lines.append(f"{'#'*78}\n")

    return lines


def nb2txt(ipynb_paths, out_path=None, merge=False):
    """
    Convert one or more Jupyter notebooks to text.

    Parameters
    ----------
    ipynb_paths : str | Path | list
        Single notebook path, or list of paths.
    out_path : str | Path | None
        Output file path. If None, uses notebook name with .txt extension.
        For merge=True, defaults to 'merged_notebooks.txt'.
    merge : bool
        If True, combine all notebooks into a single file.
        If False, create one .txt per notebook.

    Returns
    -------
    list[str]
        Paths to the created text files.
    """
    if isinstance(ipynb_paths, (str, Path)):
        ipynb_paths = [ipynb_paths]

    ipynb_paths = [Path(p) for p in ipynb_paths]

    created = []

    if merge:
        # Single merged output
        if out_path is None:
            out_path = Path("merged_notebooks.txt")
        else:
            out_path = Path(out_path)

        all_lines = []
        all_lines.append(f"{'='*78}")
        all_lines.append(f"# MERGED NOTEBOOKS CONVERSION")
        all_lines.append(f"# Total notebooks: {len(ipynb_paths)}")
        all_lines.append(f"# {'='*78}")

        for p in ipynb_paths:
            if not p.exists():
                print(f"[SKIP] Not found: {p}")
                continue
            if p.suffix != ".ipynb":
                print(f"[SKIP] Not a notebook: {p}")
                continue

            try:
                cell_lines = extract_notebook(p)
                all_lines.extend(cell_lines)
            except Exception as e:
                print(f"[ERROR] Failed to parse {p}: {e}")
                continue

        all_lines.append(f"\n{'='*78}")
        all_lines.append(f"# END OF MERGED FILE")
        all_lines.append(f"{'='*78}\n")

        with open(out_path, "w", encoding="utf-8") as f:
            f.write("\n".join(all_lines))

        print(f"[OK] Merged {len(ipynb_paths)} notebooks → {out_path}")
        created.append(str(out_path))

    else:
        # One file per notebook
        for p in ipynb_paths:
            if not p.exists():
                print(f"[SKIP] Not found: {p}")
                continue
            if p.suffix != ".ipynb":
                print(f"[SKIP] Not a notebook: {p}")
                continue

            if out_path is None:
                out = p.with_suffix(".txt")
            else:
                out = Path(out_path)

            try:
                lines = extract_notebook(p)
                with open(out, "w", encoding="utf-8") as f:
                    f.write("\n".join(lines))
                print(f"[OK] Converted: {p} → {out}")
                created.append(str(out))
            except Exception as e:
                print(f"[ERROR] Failed to convert {p}: {e}")

    return created


def main():
    parser = argparse.ArgumentParser(
        description="Convert Jupyter notebooks to clean text files"
    )
    parser.add_argument("notebooks", nargs="+", help="One or more .ipynb files")
    parser.add_argument(
        "-o",
        "--output",
        help="Output path. For single file: output name. For merge: merged file name.",
    )
    parser.add_argument(
        "-m",
        "--merge",
        action="store_true",
        help="Merge all notebooks into a single text file",
    )
    args = parser.parse_args()

    nb2txt(args.notebooks, out_path=args.output, merge=args.merge)

In [ ]:
# from nb2txt import nb2txt

# convert_files = ["RLVR_Part1_AlphaCache.ipynb", "RLVR_Part2_Training.ipynb"]
# convert_files = ["RLVR_Part1_AlphaCache_v2.ipynb"]
# convert_files = [r"..\stocks\notebooks_RLVR_v2\rl_discovery\environment.py"]
convert_files = [
    # "00_RLVR_data_process_v7.ipynb",
    # "01_RLVR_Part1_AlphaCache_v5.ipynb",
    "02_RLVR_Part2_Training_v19a.ipynb",
    # "03_RLVR_Part3_Analysis_v4.ipynb",
    # "03_text.ipynb",
]

output_filename = "02.txt"

# Merge two notebooks into one file
nb2txt(
    convert_files,
    out_path=output_filename,
    merge=True,
)

# # Or merge ALL notebooks in a directory
# from pathlib import Path

# notebooks = sorted(Path(".").glob("*.ipynb"))
# nb2txt(notebooks, out_path="all_notebooks.txt", merge=True)

[SKIP] Not found: 02_RLVR_Part2_Training_v119a.ipynb
[OK] Merged 1 notebooks → 02.txt


['02.txt']

In [ ]:
from pathlib import Path


class DisplayPath:
    """Wraps a Path so repr() shows Path("...") instead of WindowsPath("...")."""

    def __init__(self, path):
        if isinstance(path, DisplayPath):
            self._path = path._path
        elif isinstance(path, Path):
            self._path = path
        else:
            self._path = Path(path)

    def __repr__(self):
        return f'Path("{self._path.as_posix()}")'

    def __str__(self):
        return str(self._path)

    def __getattr__(self, name):
        return getattr(self._path, name)

    def __truediv__(self, other):
        return DisplayPath(self._path / other)

    def __eq__(self, other):
        if isinstance(other, DisplayPath):
            return self._path == other._path
        return self._path == other

    def __hash__(self):
        return hash(self._path)


def list_files(
    root_dir,
    exclude_dirs=None,
    exclude_exts=None,
    exclude_prefixes=None,
    exclude_names=None,
    include_names=None,  # <-- NEW
):
    root = Path(root_dir)
    exclude_dirs = set(exclude_dirs or [])
    raw_exts = set(exclude_exts or [])
    exclude_exts = {
        ext.lower() if ext.startswith(".") else f".{ext.lower()}" for ext in raw_exts
    }
    exclude_prefixes = tuple(exclude_prefixes or [])
    exclude_names = set(exclude_names or [])
    include_names = set(include_names or [])  # <-- NEW
    results = []

    for path in root.rglob("*"):
        if any(part in exclude_dirs for part in path.parts):
            continue
        if path.is_file():
            name = path.name
            if path.suffix.lower() in exclude_exts:
                continue
            if name.startswith(exclude_prefixes):
                continue
            if any(sub in name for sub in exclude_names):
                continue
            # Only include files whose name contains at least one substring from include_names
            if include_names and not any(sub in name for sub in include_names):
                continue
            results.append(DisplayPath(path))

    return results


# Usage
files = list_files(
    r"..\notebooks_RLVR_v2",
    exclude_dirs={".git", "__pycache__", "node_modules", "archive", "output", "tests"},
    exclude_exts={
        ".txt",
        ".ipynb",
        ".xlsx",
        ".md",
        ".parquet",
        ".pkl",
        "__init__",
    },
    exclude_prefixes={"__init__", "_trash"},
    exclude_names={"trash", "backup", "temp", "txt", "utils"},
    include_names={
        "environment.py",
        "builder.py",
        "registry.py",
        "oracle.py",
        "validator.py",
        "settings.py",
    },  # <-- only files with "agent" or "oracle" in the name
)

# Add/remove as needed
# files.append(DisplayPath(r"..\notebooks_RLVR_v2\extra_script.py"))
files = [f for f in files if "draft" not in f.name]

files

In [ ]:
from pathlib import Path

# # Your list of files
files = [
    Path("../notebooks_RLVR_v2/core/settings.py"),
    Path("../notebooks_RLVR_v2/data_pipeline/builder.py"),
    Path("../notebooks_RLVR_v2/rl_discovery/environment.py"),
    Path("../notebooks_RLVR_v2/rl_discovery/oracle.py"),
    Path("../notebooks_RLVR_v2/rl_discovery/trainer.py"),
    Path("../notebooks_RLVR_v2/rl_discovery/validator.py"),
    Path("../notebooks_RLVR_v2/strategy/registry.py"),
]

output_path = Path("teir_1_files.txt")

with open(output_path, "w", encoding="utf-8") as out:
    for i, f in enumerate(files, start=1):
        # ─── Clear separator with file number ───
        out.write(f"\n{'#'*80}")
        out.write(f"\n{'#'*80}\n")
        out.write(f"### FILE [{i}] : {f.name}\n")
        out.write(f"### PATH   : {f}\n")
        out.write(f"{'#'*80}\n")
        out.write(f"{'#'*80}\n\n")

        try:
            content = f.read_text(encoding="utf-8")
            out.write(content)
            # Ensure trailing newline so content doesn't bleed into next header
            if content and not content.endswith("\n"):
                out.write("\n")
        except Exception as e:
            out.write(f"[ERROR reading file: {e}]\n")

print(f"Combined {len(files)} files into: {output_path.resolve()}")

Dot-com Bubble Burst:
  Calculated Start: 2000-03-27
  Calculated End: 2006-10-26
  Max Drawdown: -47.52%
  Duration (Days): 2404

2008 Financial Crisis:
  Calculated Start: 2007-10-10
  Calculated End: 2012-08-16
  Max Drawdown: -55.19%
  Duration (Days): 1772

COVID-19 Crash:
  Calculated Start: 2020-02-20
  Calculated End: 2020-08-10
  Max Drawdown: -33.72%
  Duration (Days): 172